In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig


d:\desktop\Graduation Project\GRAD_PROJECT\invoice-ai-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pandas as pd

In [4]:
model_save_path = r"D:\desktop\\Graduation Project\\GRAD_PROJECT\\invoice-ai-project\\ai_core\\models\\Training_model"

instruct_model = AutoModelForSeq2SeqLM.from_pretrained(model_save_path)
instruct_tokenizer = AutoTokenizer.from_pretrained(model_save_path)

In [2]:
words_img = pd.read_csv("000010.csv")

In [3]:
words_img.head()

,top_left_x,top_right_x,bottom_right_x,bottom_left_x,top_left_y,top_right_y,bottom_right_y,bottom_left_y,text,confidence
0,883,1039,1039,883,101,101,163,163,فاتورة,0.991548
1,2274,2497,2497,2274,125,125,190,190,ما كدونالدز,0.587669
2,887,1227,1227,887,180,180,223,223,رقم الفاتورة: فاثورة -20626,0.779752
3,2299,2435,2435,2299,205,205,243,243,الملك فود,0.574205
4,885,1229,1229,885,231,231,273,273,تاريخ الفاتورة: 05-08-2025,0.925067


In [4]:
text = words_img.text
text

0                          فاتورة
1                     ما كدونالدز
2     رقم الفاتورة: فاثورة -20626
3                       الملك فود
4      تاريخ الفاتورة: 05-08-2025
5                    فاتورة إلى :
6                            شركة
7                 التحلية. الرياض
8                        الإجمالي
9                         التملفة
10                          الحمة
11                         الخدمة
12                      36.6 ريال
13                      18.3 ريال
14                              2
15                           قهوة
16                     55.02 ريال
17                      18.34 رال
18                              3
19                     قووة لاتيه
20                    157.16 ريال
21                     52.39 ريال
22                              3
23                      بيترا حجم
24                    248.78 ريال
25                       الإجمالي
26                         معنا .
27                          شكراً
28                     37.32 ريال
29            

In [23]:
instruct_model_corrections = []

for distorted_text in text:
    prompt = f"""
صحح الكلمة أو الجملة التالية:

{distorted_text}

الجملة المصححة:
"""
    input_ids = instruct_tokenizer(prompt, return_tensors="pt").input_ids

    

    instruct_model_outputs = instruct_model.generate(
        input_ids=input_ids,
        generation_config=GenerationConfig(max_new_tokens=200)
    )
    instruct_model_text_output = instruct_tokenizer.decode(instruct_model_outputs[0], skip_special_tokens=True)
    instruct_model_corrections.append(instruct_model_text_output)

results_df = pd.DataFrame({
    'distorted_text': text,
    'instruct_model_correction': instruct_model_corrections
})


print(results_df)


    distorted_text                          instruct_model_correction
0           الأمإن  صحح الكلمة أو الجملة التالية : الأم إن الجملة ...
1        هف:شداد ٣  صحح الكلمة أو الجملة التالية : هف : شداد ، ٣ ا...
2          ٥٧٥٨٤٤٥  صحح الكلمة أو الجملة التالية : ٥٧٥٨٤٤٥ الجملة ...
3   لحلول الأعسمال  صحح الكلمة أو الجملة التالية : لحلول الأعسمال ...
4    ٥٨5ز٥٧؟ 55زء   صحح الكلمة أو الجملة التالية : ٥٨5 ، ز٥٧ ؟ 55 ...
..             ...                                                ...
63            لاصق  صحح الكلمة أو الجملة التالية : لاصق الجملة الم...
64           أحبار  صحح الكلمة أو الجملة التالية : أحبار الجملة ال...
65             عبو  صحح الكلمة أو الجملة التالية : عبو الجملة المص...
66         المجموع  صحح الكلمة أو الجملة التالية : المجموع الجملة ...
67         المجموع  صحح الكلمة أو الجملة التالية : المجموع الجملة ...

[68 rows x 2 columns]


In [24]:
results_df.to_csv("test.csv",index=False, encoding="utf-8-sig")

In [15]:
import os
import re
import json
import Levenshtein
import pandas as pd

VOCAB_PATH = 'vocabs.json'

def load_vocab_from_json(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            vocab = data
            print(f"Loaded vocab from JSON. Number of words: {len(vocab)}")
            return set(vocab)
    else:
        print("Vocab file not found.")
        return set()

jamid_names = load_vocab_from_json(VOCAB_PATH)

if jamid_names:
    print("Sample of vocab words:")
    for name in list(jamid_names)[:20]:
        print(name)
else:
    print("No vocab words loaded.")

def preprocess_arabic_word(word):
    word = re.sub(r'[\u064B-\u065F]', '', word)
    word = word.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا')
    return word

def correct_word(word, correct_words):
    word = preprocess_arabic_word(word)
    if word in correct_words:
        return word

    suggestions = []
    for correct_word in correct_words:
        clean_correct_word = preprocess_arabic_word(correct_word)
        distance = Levenshtein.distance(word, clean_correct_word)
        if distance <= 3:
            suggestions.append((correct_word, distance))

    if suggestions:
        suggestions.sort(key=lambda x: x[1])
        return suggestions[0][0]
    else:
        return word


Loaded vocab from JSON. Number of words: 291
Sample of vocab words:
مبرد
داخلية
خزانة
فاتورة
كاس
دعم
أقلام
ريال
التحتية
عبر
الملك
كمية
لابتوب
المكرمة
رقم
للبيانات
أبيض
مدة
الاستثمارات
لاتيه


In [21]:
import pandas as pd

# دالة لتصحيح كل كلمة داخل النص
def correct_text_column(text, correct_words):
    if pd.isna(text):
        return text

    # تقسيم النص إلى كلمات عربية أو أرقام صحيحة/عشرية مع الحفاظ عليها كما هي
    tokens = re.findall(r'[\u0600-\u06FF]+|\d+(?:[.,-]\d+)?', text)

    corrected_tokens = []
    for token in tokens:
        if re.fullmatch(r'[\u0600-\u06FF]+', token):  
            # إذا كانت الكلمة عربية -> نصححها
            corrected_tokens.append(correct_word(token, correct_words))
        else:
            # أرقام (حتى العشرية) -> نتركها كما هي
            corrected_tokens.append(token)

    return " ".join(corrected_tokens)

# تطبيق التصحيح على العمود
words_img['corrected_text'] = words_img['text'].apply(lambda x: correct_text_column(x, jamid_names))

# حفظ النتائج في ملف جديد
words_img.to_csv("corrected_text_output.csv", index=False, encoding='utf-8-sig')

# عرض عينة من النتائج
print(words_img[['text', 'corrected_text']].head(10))


                          text             corrected_text
0                       فاتورة                     فاتورة
1                  ما كدونالدز             ماء ماكدونالدز
2  رقم الفاتورة: فاثورة -20626  رقم الفاتورة فاتورة 20626
3                    الملك فود                  الملك فهد
4   تاريخ الفاتورة: 05-08-2025  تاريخ الفاتورة 05-08 2025
5                 فاتورة إلى :                 فاتورة إلى
6                         شركة                       شركة
7              التحلية. الرياض             التحلية الرياض
8                     الإجمالي                   الإجمالي
9                      التملفة                    التكلفة


In [22]:
words_img

,top_left_x,top_right_x,bottom_right_x,bottom_left_x,top_left_y,top_right_y,bottom_right_y,bottom_left_y,text,confidence,corrected_text
0,883,1039,1039,883,101,101,163,163,فاتورة,0.991548,فاتورة
1,2274,2497,2497,2274,125,125,190,190,ما كدونالدز,0.587669,ماء ماكدونالدز
2,887,1227,1227,887,180,180,223,223,رقم الفاتورة: فاثورة -20626,0.779752,رقم الفاتورة فاتورة 20626
3,2299,2435,2435,2299,205,205,243,243,الملك فود,0.574205,الملك فهد
4,885,1229,1229,885,231,231,273,273,تاريخ الفاتورة: 05-08-2025,0.925067,تاريخ الفاتورة 05-08 2025
5,2333,2494,2494,2333,389,389,431,431,فاتورة إلى :,0.581766,فاتورة إلى
6,2408,2497,2497,2408,470,470,519,519,شركة,0.990923,شركة
7,2228,2428,2428,2228,524,524,573,573,التحلية. الرياض,0.771667,التحلية الرياض
8,1236,1333,1333,1236,667,667,703,703,الإجمالي,0.459695,الإجمالي
9,1648,1736,1736,1648,670,670,696,696,التملفة,0.262003,التكلفة
